# Simple Data Preprocessing Notebook
This notebook demonstrates basic preprocessing.

# Use the following function to access documentation for any function or class:
```python
help(function_or_class_name)
dir(function_or_class_name)
```

help function will show the docstring, while dir will list attributes and methods. This is useful for exploring new libraries or functions.

**Q:** Import required libraries and generate the synthetic dataset (single cell).

In [1]:
# Imports and data generation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import os

# Reproducibility and size
np.random.seed(42)
n = 150_000

# Generate features
heights = np.random.normal(165, 10, size=n).astype(float)  # Heights in cm
weights = np.random.normal(70, 15, size=n).astype(float)  # Weights in kg
bmi = weights / ((heights / 100) ** 2)  # BMI calculation
regions = np.random.choice(['North', 'South', 'East', 'West'], size=n, p=[0.25, 0.25, 0.25, 0.25])
activity_level = np.random.choice(['Low', 'Medium', 'High'], size=n, p=[0.3, 0.5, 0.2])
# Target correlated with BMI and activity level
target = ((bmi > 25) & (activity_level == 'Low')).astype(int)

# Build DataFrame
df = pd.DataFrame({
    'height': heights,
    'weight': weights,
    'bmi': bmi,
    'region': regions,
    'activity_level': activity_level,
    'target': target
})

# Introduce missingness
mask = np.random.rand(n) < 0.07
df.loc[mask, 'weight'] = np.nan
mask = np.random.rand(n) < 0.05
df.loc[mask, 'region'] = np.nan
mask = np.random.rand(n) < 0.04
df.loc[mask, 'activity_level'] = np.nan

print('Done: imports and data generation completed; df shape =', df.shape)

Done: imports and data generation completed; df shape = (150000, 6)


**Q:** Check null value counts in the DataFrame.

In [2]:
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

weight            10340
region             7442
activity_level     5989
dtype: int64


In [4]:
assert isinstance(null_counts, pd.Series)
print('OK: null counts computed')

OK: null counts computed


**Q:** Check duplicate row count in the DataFrame.

In [3]:
dup_count = df.duplicated().sum()
print('Duplicate rows:', dup_count)

Duplicate rows: 0


In [5]:
assert isinstance(dup_count, (int, np.integer))
print('OK: duplicate count computed')

OK: duplicate count computed


**Q:** Remove rows with any null values  .

In [6]:
before = df.shape[0]
df = df.dropna()
after = df.shape[0]
print('Rows before:', before, 'after dropna:', after, 'removed:', before - after)

Rows before: 150000 after dropna: 127410 removed: 22590


In [7]:
assert after <= before
print('OK: dropna applied')

OK: dropna applied


**Q:** Remove duplicate rows  .

In [8]:
before_dup = df.shape[0]
df = df.drop_duplicates()
after_dup = df.shape[0]
print('Rows before dedup:', before_dup, 'after dedup:', after_dup, 'removed:', before_dup - after_dup)

Rows before dedup: 127410 after dedup: 127410 removed: 0


In [9]:
assert after_dup <= before_dup
print('OK: duplicates removed')

OK: duplicates removed


**Q:** Separate features and target into `X` and `y`.

In [10]:
X = df.drop('target', axis=1)
y = df['target']
print('X shape:', X.shape, 'y shape:', y.shape)

X shape: (127410, 5) y shape: (127410,)


In [11]:
assert X.shape[0] == y.shape[0]
print('OK: X and y separated')

OK: X and y separated


**Q:** Can you find the number of people with height greater than 180 cm?


In [12]:
people_over_180 = (df['height'] > 180).sum()
print('Number of people with height > 180 cm:', people_over_180)

Number of people with height > 180 cm: 8611


In [ ]:
assert isinstance(people_over_180, (int, np.integer))
print('OK: people over 180 cm counted')

**Q:** Perform training and testing split with 80/20 split and random state as 42.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

Train shape: (101928, 5) Test shape: (25482, 5)


In [ ]:
assert X_train.shape[0] + X_test.shape[0] == X.shape[0]
print('OK: train/test split done')

**Q:** Convert categorical features to numeric using one hot encoding.

In [14]:
X_train_encode = pd.get_dummies(X_train, columns=['region', 'activity_level'], drop_first=True)
X_test_encode = pd.get_dummies(X_test, columns=['region', 'activity_level'], drop_first=True)
print('One-hot encoding applied. Train shape:', X_train_encode.shape, 'Test shape:', X_test_encode.shape)

One-hot encoding applied. Train shape: (101928, 8) Test shape: (25482, 8)


In [ ]:
#check the column are encoded or not using assert
expected_city_cols = [ 'region_North', 'region_South']
expected_activity_cols = ['activity_level_Medium', ]
for col in expected_city_cols + expected_activity_cols:
    assert col in X_train_encode.columns, f'Missing column: {col}'
    assert col in X_test_encode.columns, f'Missing column: {col}'
print('OK: one-hot encoding columns verified')


**Q:** Train a Decision Tree and evaluate accuracy.

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_encode, y_train)
print('Model trained')

Model trained


**Q:** Calculate the accuracy of the model on the test set.

In [16]:
preds = model.predict(X_test_encode)
acc = accuracy_score(y_test, preds)


In [17]:
print(f'Accuracy: {acc:.4f}')
assert 0.0 <= acc <= 1.0
print('OK: accuracy computed')

Accuracy: 1.0000
OK: accuracy computed
